# 02 - Text Pipeline: Final XLM-RoBERTa Training & Retrieval Pipeline

**Stage:** Text-based fake news detection — final model

**What this notebook does:**
- Loads and merges the raw news dataset, cleans text (links, emojis, hashtags, mentions)
- Encodes labels and fine-tunes `xlm-roberta-large` for binary classification (real/fake)
- Builds a semantic retrieval step (Sentence-BERT + cosine similarity) to fetch the most similar verified news
- Assembles the end-to-end fake news detection pipeline (classification + credibility scoring + evidence retrieval)

**Input:** raw scraped news dataset
**Output:** the final fine-tuned text classification model + retrieval pipeline, consumed by `03_text_api_deployment.ipynb`

> Part of the WAVE project — AI section (text-based fake news detection).

# **تحميل البيانات ودمج النصوص في عمود موحد text**

In [ ]:
# --- Environment configuration ---
# Set BASE_DIR to wherever your data/model checkpoints live locally.
# If running on Google Colab, uncomment the two lines below.
# from google.colab import drive
# drive.mount('/content/drive')

import os
BASE_DIR = os.environ.get("WAVE_DATA_DIR", "./data")


In [ ]:
import pandas as pd

df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")
df

In [ ]:

df["Arabic Text"] = df["Arabic Text"].fillna("")
df["English Text"] = df["English Text"].fillna("")


df["text"] = df["Arabic Text"] + df["English Text"]


df = df[df["text"].str.strip() != ""]

df[["text", "label"]]

# **تنظيف النصوص داخل عمود text**

إزالة:

الإيموجيات

الهاشتاغات

الروابط

mentions (@)

الرموز الزائدة

مع الحفاظ على:

الأرقام

التواريخ

النص المهم

In [ ]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""


    text = re.sub(r"["
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  "]+", '', text)


    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)

  
    text = re.sub(r"#\w+", '', text)
    text = re.sub(r"@\w+", '', text)

  
    text = re.sub(r"[•●▪️⭐️➤✓✔️▶️🔴🟢]", '', text)

    
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df["text"] = df["text"].apply(clean_text)


df = df[df["text"].str.strip() != ""]


df[["text", "label"]]


# **ترميز التصنيفات النصية إلى أرقام**

تحويل عمود label ليكون:

real → 1

fake → 0

In [ ]:

label_map = {"real": 1, "fake": 0}
df["label"] = df["label"].map(label_map)


df = df[df["label"].isin([0, 1])]


df["label"].value_counts()
df

# **احفظ نسخه نهائيه مشان ماعيد تجهيز الداتا**

In [ ]:

df = df[["text", "label", "Source", "Arabic Text", "English Text"]]


df.to_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv", index=False, encoding="utf-8-sig")

print(" تم حفظ النسخة النهائية في: BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")
print("عدد العينات النهائية:", len(df))


# some important Plots

# هل الفئة الفيك أكثر من الحقيقية؟

# هل الأخبار طويلة أم قصيرة؟ وهل تختلف بين الفئتين؟

# ما هو الحد الأقصى للطول المناسب للنموذج؟
# هي الاسئله بعرف جوابها من خلال الرسمات

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(5,4))
sns.countplot(x="label", data=df)
plt.title("Class Distribution (0 = Fake, 1 = Real)")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()


df["word_count"] = df["text"].apply(lambda x: len(x.split()))
plt.figure(figsize=(6,4))
sns.histplot(df["word_count"], bins=40, kde=True)
plt.title("Distribution of Text Lengths (Words)")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()


plt.figure(figsize=(5,4))
sns.boxplot(x="label", y="word_count", data=df)
plt.title("Text Length by Class")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Word Count")
plt.show()


(Class Distribution)
ممتاز جدًا  البيانات متوازنة تقريبًا بين real و fake.

لا حاجة لتطبيق oversampling أو undersampling الآن.

توزيع أطوال النصوص
معظم النصوص تقع بين 100 إلى 400 كلمة

بعض النصوص تتجاوز 2000 كلمة، وهناك outliers تصل لـ 16,000 كلمة!

وهذا مهم جدًا لتحديد max_length أثناء التوكنزة.

Boxplot حسب الفئة
يظهر أن الأخبار الحقيقية غالبًا أطول قليلًا من الأخبار المزيفة.

ما زال هناك تفاوت كبير بالطول.

# **تجهيز النموذج XLM-Roberta و Tokenizer**

# تحميل النموذج xlm-roberta-large والـ Tokenizer

In [ ]:
!pip install -q transformers datasets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


#model_name = "xlm-roberta-large"
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(" تم تحميل XLM-Roberta-Large بنجاح")


# **تقسيم البيانات وتحويلها إلى DatasetDict جاهز للتدريب**

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")
df = df[["text", "label"]]


train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)


dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

print(" تم تجهيز DatasetDict للتدريب")


In [ ]:

dataset.save_to_disk("BASE_DIR/Senior Data/fake_news_tokenized_dataset")

# **من هون بكمل اذا فصلت السيشن مافي داعي اعيد اللي قبل**

In [ ]:
from datasets import load_from_disk
dataset = load_from_disk("BASE_DIR/Senior Data/fake_news_tokenized_dataset")

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512  
    )

# 
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=128,               
    remove_columns=["text"],
    load_from_cache_file=False,   
    keep_in_memory=False         
)


tokenized_datasets.set_format("torch")


import gc
del dataset        
gc.collect()       

print(" تم تنفيذ التوكنزة بكفاءة على دفعات وتقليل استهلاك الذاكرة.")

In [ ]:
# 💾 حفظ البيانات المرمّزة بصيغة HuggingFace Dataset
tokenized_datasets.save_to_disk("BASE_DIR/Senior Data/fake_news_tokenized_dataset2")

print(" تم حفظ tokenized_datasets في Google Drive بنجاح")


In [ ]:
from datasets import load_from_disk
tokenized_datasets = load_from_disk("BASE_DIR/Senior Data/fake_news_tokenized_dataset2")


In [ ]:
print(tokenized_datasets)

In [ ]:

print(tokenized_datasets["train"][0])


In [ ]:
print("Train size:", len(tokenized_datasets["train"]))
print("Validation size:", len(tokenized_datasets["validation"]))
print("Test size:", len(tokenized_datasets["test"]))


# **some important plots**

# توزيع أطوال input_ids
## Boxplot لطول التوكنات حسب الفئة (real/fake)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch


lengths = [torch.sum(x["attention_mask"]).item() for x in tokenized_datasets["train"]]
labels = [x["label"] for x in tokenized_datasets["train"]]


plt.figure(figsize=(7,4))
sns.histplot(lengths, bins=50, kde=True)
plt.title("Distribution of Tokenized Text Lengths (Train Set)")
plt.xlabel("Number of tokens (non-padded)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


plt.figure(figsize=(6,4))
sns.boxplot(x=labels, y=lengths)
plt.title("Token Length by Label (0 = Fake, 1 = Real)")
plt.xlabel("Label")
plt.ylabel("Token Length")
plt.grid(True)
plt.show()


# **عداد TrainingArguments**

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/xlm_roberta_fakenews",      
    evaluation_strategy="steps",                   
    save_strategy="steps",
    eval_steps=500,                                 
    save_steps=500,                                
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,                            
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",                   
    greater_is_better=True,
    logging_dir="/content/logs",
    logging_strategy="steps",
    logging_steps=100,
    fp16=True                                        
)




In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
from transformers import Trainer, EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # سيتوقف إذا لم يتحسن F1 بعد 2 تقييمات
)


In [ ]:
train_result = trainer.train()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df_log = pd.DataFrame(trainer.state.log_history)


df_log = df_log[df_log["step"].notna()]


df_val = df_log[df_log["eval_loss"].notna()]
df_train = df_log[df_log["loss"].notna()]


plt.figure(figsize=(10,5))
plt.plot(df_train["step"], df_train["loss"], label="Training Loss", color="orange", linestyle='--', marker='o')
plt.plot(df_val["step"], df_val["eval_loss"], label="Validation Loss", color="blue", marker='s')
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss (Smoothed & Balanced)")
plt.legend()
plt.grid(True)
plt.show()


# **Testing on unseen data**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch


predictions = trainer.predict(tokenized_datasets["test"])
y_pred = torch.argmax(torch.tensor(predictions.predictions), axis=1)
y_true = torch.tensor(predictions.label_ids)


print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


In [ ]:

save_path = "BASE_DIR/xlm_roberta_fakenews_best"
trainer.save_model(save_path)            
tokenizer.save_pretrained(save_path)   

print(f"✅ تم حفظ النموذج في: {save_path}")


# **Pipeline Fake News Detection**

In [ ]:
!pip install langdetect
from langdetect import detect


In [ ]:
import pandas as pd
from langdetect import detect


def detect_language(text):
    try:
        lang = detect(text)
        return 'ar' if lang == 'ar' else 'en'
    except:
        return 'unknown'


df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")


df = df[["Arabic Text", "English Text", "Source", "label"]]


arabic_texts = df[df["Arabic Text"].notna() & (df["Arabic Text"].str.strip() != "")]
english_texts = df[df["English Text"].notna() & (df["English Text"].str.strip() != "")]


In [ ]:
arabic_texts

In [ ]:
english_texts

In [ ]:
from sentence_transformers import SentenceTransformer
import torch


embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


english_embeddings = embedder.encode(
    english_texts["English Text"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)


arabic_embeddings = embedder.encode(
    arabic_texts["Arabic Text"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)


torch.save(english_embeddings, "BASE_DIR/embedded data/english_embeddings.pt")
torch.save(arabic_embeddings, "BASE_DIR/embedded data/arabic_embeddings.pt")


english_texts.to_csv("BASE_DIR/embedded data/english_texts.csv", index=False)
arabic_texts.to_csv("BASE_DIR/embedded data/arabic_texts.csv", index=False)

print(" تم استخراج وحفظ الـ embeddings والنصوص بنجاح.")


In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util


arabic_texts = pd.read_csv("BASE_DIR/embedded data/arabic_texts.csv")
english_texts = pd.read_csv("BASE_DIR/embedded data/english_texts.csv")


arabic_embeddings = torch.load("BASE_DIR/embedded data/arabic_embeddings.pt")
english_embeddings = torch.load("BASE_DIR/embedded data/english_embeddings.pt")


embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


In [ ]:
from langdetect import detect
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sentence_transformers import util


model_path = "BASE_DIR/xlm_roberta_fakenews_best"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)


embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


arabic_texts = pd.read_csv("BASE_DIR/embedded data/arabic_texts.csv")
english_texts = pd.read_csv("BASE_DIR/embedded data/english_texts.csv")
arabic_embeddings = torch.load("BASE_DIR/embedded data/arabic_embeddings.pt")
english_embeddings = torch.load("BASE_DIR/embedded data/english_embeddings.pt")


def detect_language(text):
    try:
        lang = detect(text)
        return 'ar' if lang == 'ar' else 'en'
    except:
        return 'unknown'


def classify_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).detach().numpy()[0]
    pred_label = int(probs.argmax())
    label_name = "Real" if pred_label == 1 else "Fake"
    credibility_score = round(float(probs[pred_label]), 3)
    return label_name, credibility_score


def retrieve_similar_news(input_text, lang, top_k=5):
    query_embedding = embedder.encode(input_text, convert_to_tensor=True)
    if lang == 'ar':
        embeddings = arabic_embeddings
        target_df = arabic_texts
    else:
        embeddings = english_embeddings
        target_df = english_texts

    similarities = util.pytorch_cos_sim(query_embedding, embeddings)[0]
    top_results = torch.topk(similarities, k=top_k)

    similar_news = []
    for idx in top_results.indices:
        idx = int(idx)
        text = target_df.iloc[idx]["Arabic Text"] if lang == "ar" else target_df.iloc[idx]["English Text"]
        source = target_df.iloc[idx]["Source"]
        label = "Real" if target_df.iloc[idx]["label"] == 1 else "Fake"
        score = round(float(similarities[idx]), 3)
        similar_news.append((text, source, label, score))

    return similar_news



def pipeline(user_input):
    lang = detect_language(user_input)
    print(" الخبر المُدخل:")
    print(user_input)
    print(f"\n اللغة: {'Arabic' if lang == 'ar' else 'English'}")

    label, credibility = classify_news(user_input)
    print(f"\n التصنيف: {label}")
    print(f" Credibility Score: {credibility}")
    print("🟢" if credibility >= 0.9 else "🟡" if credibility >= 0.7 else "🔴")

    print("\n📰 الأخبار المشابهة:")
    similar = retrieve_similar_news(user_input, lang)
    for i, (text, source, label, score) in enumerate(similar, 1):
        print(f"\n🔹 {i}. المصدر: {source}")
        print(f"   التصنيف: {label} | Score: {score}")
        print(f"   الخبر: {text[:200]}...")  


In [ ]:
user_news = "الرئيس احمد الشرع يهدد اسرائيل"
pipeline(user_news)


In [ ]:
import pandas as pd


df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")


source_counts = df['Source'].value_counts()


print("📰 قائمة المصادر وعدد مرات الظهور:\n")
print(source_counts)


In [ ]:
import pandas as pd
import torch

df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")


arabic_texts = pd.read_csv("BASE_DIR/embedded data/arabic_texts.csv")
english_texts = pd.read_csv("BASE_DIR/embedded data/english_texts.csv")


arabic_embeddings = torch.load("BASE_DIR/embedded data/arabic_embeddings.pt")
english_embeddings = torch.load("BASE_DIR/embedded data/english_embeddings.pt")

print(" تم تحميل البيانات والـ embeddings بنجاح.")


In [ ]:
arabic_texts

In [ ]:
!pip install langdetect

إذا كانت النتيجة Fake:

نبحث عن أقرب خبر مصحح (label = 1) بنفس اللغة وتشابه ≥ 0.8

نعرضه مع مصدره والمصداقية

إذا لم يوجد → نطبع "لا يوجد خبر مصحح بثقة عالية"

إذا كانت النتيجة Real:

نطبع أقرب خبر مشابه (من النتائج المسترجعة سابقًا)

مع مصدره ودرجة الثقة



In [ ]:

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from langdetect import detect


df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")
arabic_texts = pd.read_csv("BASE_DIR/embedded data/arabic_texts.csv")
english_texts = pd.read_csv("BASE_DIR/embedded data/english_texts.csv")
arabic_embeddings = torch.load("BASE_DIR/embedded data/arabic_embeddings.pt")
english_embeddings = torch.load("BASE_DIR/embedded data/english_embeddings.pt")


source_reliability = {
    "No Source": 0.1,
    "Syria TV": 0.95,
    "Al Jazeera": 0.85,
    "Free Syria News Network": 0.9,
    "Takkad": 0.95,
    "Audio Transcripts": 0.6,
    "Syriana Educational": 0.9
}
def get_source_score(source_name):
    return source_reliability.get(source_name, 0.4)

def calculate_credibility(similarity_score, source_name):
    source_score = get_source_score(source_name)
    return 0.6 * similarity_score + 0.4 * source_score


embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def detect_language(text):
    try:
        lang = detect(text)
        return 'ar' if lang == 'ar' else 'en'
    except:
        return 'unknown'

def get_input_embedding(text):
    lang = detect_language(text)
    embedding = embedder.encode(text, convert_to_tensor=True)
    return embedding, lang


model_path = "BASE_DIR/xlm_roberta_fakenews_best"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def classify_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0][predicted_class].item()
    label = "Real" if predicted_class == 1 else "Fake"
    return label, round(confidence, 4)


def retrieve_top_similar_news(input_embedding, language):
    if language == 'ar':
        embeddings = arabic_embeddings
        texts_df = arabic_texts
        text_column = 'Arabic Text'
    else:
        embeddings = english_embeddings
        texts_df = english_texts
        text_column = 'English Text'
    similarities = F.cosine_similarity(input_embedding, embeddings)
    top_k = torch.topk(similarities, k=3)
    top_indices = top_k.indices.tolist()
    top_scores = top_k.values.tolist()
    top_results = []
    for idx, sim_score in zip(top_indices, top_scores):
        row = texts_df.iloc[idx]
        text = row[text_column]
        source = row['Source']
        credibility = calculate_credibility(sim_score, source)
        top_results.append({
            'text': text,
            'source': source,
            'similarity': round(sim_score, 4),
            'credibility': round(credibility, 4)
        })
    return top_results


def show_final_output(user_input, input_embedding, lang, top_matches, predicted_label):
    print("\n==============================")
    print(" Input News:")
    print(user_input)
    print("==============================")

    if predicted_label == "Fake":
        print("\n Classified as: FAKE")

        if lang == 'ar':
            real_df = arabic_texts[arabic_texts['label'] == 1].reset_index(drop=True)
            real_embeddings = arabic_embeddings[real_df.index]
            text_column = 'Arabic Text'
        else:
            real_df = english_texts[english_texts['label'] == 1].reset_index(drop=True)
            real_embeddings = english_embeddings[real_df.index]
            text_column = 'English Text'

        similarities = F.cosine_similarity(input_embedding, real_embeddings)
        max_score, max_idx = torch.max(similarities, dim=0)
        if max_score.item() >= 0.8:
            row = real_df.iloc[max_idx.item()]
            corrected_text = row[text_column]
            corrected_source = row["Source"]
            credibility = calculate_credibility(max_score.item(), corrected_source)
            print("\n Suggested Correct News Found:")
            print(f" Full Text:\n{corrected_text}")
            print(f" Source: {corrected_source}")
            print(f" Similarity: {round(max_score.item(), 4)}")
            print(f" Credibility Score: {round(credibility, 4)}")
        else:
            print("\n❌ No high-confidence corrected news found.")

    else:
        print("\n Classified as: REAL")
        most_similar = top_matches[0]
        print("\n Closest Matching News:")
        print(f" Full Text:\n{most_similar['text']}")
        print(f" Source: {most_similar['source']}")
        print(f" Similarity: {most_similar['similarity']}")
        print(f" Credibility Score: {most_similar['credibility']}")

    print("\n Top 3 Similar News:")
    for i, match in enumerate(top_matches, 1):
        print(f"\n🔹 Match #{i}:")
        print(f" Full Text:\n{match['text']}")
        print(f" Source: {match['source']}")
        print(f" Similarity: {match['similarity']}")
        print(f" Credibility Score: {match['credibility']}")

# 🧾 Run Pipeline
user_input = input("Enter a news text: ")
input_embedding, lang = get_input_embedding(user_input)
label, confidence = classify_news(user_input)
top_matches = retrieve_top_similar_news(input_embedding, lang)

print(f"\n Classification: {label}")
print(f" Confidence: {confidence}")
show_final_output(user_input, input_embedding, lang, top_matches, label)


# *FastAPI + ngrok*

In [ ]:
# ✅ Install necessary libraries
!pip install fastapi uvicorn nest-asyncio pyngrok sentence-transformers scikit-learn pandas pillow

# ✅ Add your ngrok authtoken (from https://dashboard.ngrok.com/get-started/setup)
# Replace YOUR_TOKEN_HERE with your actual token
!ngrok config add-authtoken $NGROK_AUTHTOKEN  # set this env var yourself — never hardcode tokens


In [ ]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from langdetect import detect
import pandas as pd

app = FastAPI()

# === Load Data ===
arabic_texts = pd.read_csv("BASE_DIR/embedded data/arabic_texts.csv")
english_texts = pd.read_csv("BASE_DIR/embedded data/english_texts.csv")
arabic_embeddings = torch.load("BASE_DIR/embedded data/arabic_embeddings.pt")
english_embeddings = torch.load("BASE_DIR/embedded data/english_embeddings.pt")

model_path = "BASE_DIR/xlm_roberta_fakenews_best"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

source_reliability = {
    "No Source": 0.1,
    "Syria TV": 0.95,
    "Al Jazeera": 0.85,
    "Free Syria News Network": 0.9,
    "Takkad": 0.95,
    "Audio Transcripts": 0.6,
    "Syriana Educational": 0.9
}

def detect_language(text):
    try:
        return 'ar' if detect(text) == 'ar' else 'en'
    except:
        return 'unknown'

def get_embedding(text):
    return embedder.encode(text, convert_to_tensor=True)

def get_source_score(name):
    return source_reliability.get(name, 0.4)

def calculate_credibility(similarity_score, source_name):
    return 0.6 * similarity_score + 0.4 * get_source_score(source_name)

def classify_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=1)
        pred = torch.argmax(probs).item()
        confidence = probs[0][pred].item()
    return ("Real" if pred == 1 else "Fake"), round(confidence, 4)

def retrieve_similar(input_embedding, lang):
    if lang == 'ar':
        df, emb, col = arabic_texts, arabic_embeddings, 'Arabic Text'
    else:
        df, emb, col = english_texts, english_embeddings, 'English Text'
    similarities = F.cosine_similarity(input_embedding, emb)
    top_k = torch.topk(similarities, k=3)
    results = []
    for idx, sim in zip(top_k.indices.tolist(), top_k.values.tolist()):
        row = df.iloc[idx]
        credibility = calculate_credibility(sim, row['Source'])
        results.append({
            "text": row[col],
            "source": row["Source"],
            "similarity": round(sim, 4),
            "credibility": round(credibility, 4)
        })
    return results

class NewsRequest(BaseModel):
    text: str

@app.post("/predict")
def predict(req: NewsRequest):
    text = req.text
    lang = detect_language(text)
    embedding = get_embedding(text)
    label, confidence = classify_news(text)
    similar = retrieve_similar(embedding, lang)
    return {
        "input_text": text,
        "language": lang,
        "classification": label,
        "confidence": confidence,
        "top_3_similar": similar
    }


In [ ]:
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 --reload &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)
print("🔗 Your FastAPI URL:", public_url)
